# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm—not just to build a predictive model based on historical sales data, but to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.

## Data Description

The data contains the different attributes of the various products and stores.The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product, each identifier having two letters at the beginning followed by a number.
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content of each product like low sugar, regular and no sugar
- **Product_Allocated_Area** - ratio of the allocated display area of each product to the total display area of all the products in a store
- **Product_Type** - broad category for each product like meat, snack foods, hard drinks, dairy, canned, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, others
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size of the store depending on sq. feet like high, medium and low
- **Store_Location_City_Type** - type of city in which the store is located like Tier 1, Tier 2 and Tier 3. Tier 1 consists of cities where the standard of living is comparatively higher than its Tier 2 and Tier 3 counterparts.
- **Store_Type** - type of store depending on the products that are being sold there like Departmental Store, Supermarket Type 1, Supermarket Type 2 and Food Mart
- **Product_Store_Sales_Total** - total revenue generated by the sale of that particular product in that particular store


# Installing and Importing Libraries

In [ ]:
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.34.0 -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import requests
from huggingface_hub import login, HfApi, create_repo

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
sns.set_palette('muted')

print('Ready.')

# Loading the Dataset

In [ ]:
kart = pd.read_csv('SuperKart.csv')
data = kart.copy()
print(f'{data.shape[0]:,} rows × {data.shape[1]} columns')

# Data Overview

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
print(f'{data.shape[0]:,} rows, {data.shape[1]} columns')

In [ ]:
data.info()

In [ ]:
data.describe(include='all').T

In [ ]:
# Zero nulls anywhere
data.isnull().sum()

In [ ]:
data.duplicated().sum()

**Data integrity audit:**

The dataset is structurally clean — zero nulls across all 12 columns, zero duplicate rows.
`Product_Id` carries 8,763 unique values (one per row), confirming it is a surrogate key with no repeating entities. Dropping it is non-negotiable.

The four store-level descriptor columns — `Store_Establishment_Year`, `Store_Size`, `Store_Location_City_Type`, `Store_Type` — are entirely deterministic from `Store_Id`: each of the 4 stores has exactly one value per attribute with no variation. These columns encode the *same* four-group block structure as `Store_Id` itself. Including all five simultaneously in a model creates a perfect redundancy; encoding any one of them completely captures the information in the others.

In [ ]:
# Verify perfect determinism of store attributes
for col in ['Store_Establishment_Year','Store_Size','Store_Location_City_Type','Store_Type']:
    max_unique = data.groupby('Store_Id')[col].nunique().max()
    print(f'{col}: max distinct values within any Store_Id = {max_unique}')

In [ ]:
# The 4 store profiles in full
data.drop_duplicates('Store_Id')[
    ['Store_Id','Store_Type','Store_Location_City_Type','Store_Size','Store_Establishment_Year']
].set_index('Store_Id').sort_index()

# Exploratory Data Analysis

## Univariate Analysis

In [ ]:
def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    f, (ax_box, ax_hist) = plt.subplots(
        nrows=2, sharex=True,
        gridspec_kw={'height_ratios': (0.25, 0.75)}, figsize=figsize
    )
    sns.boxplot(data=data, x=feature, ax=ax_box, showmeans=True, color='steelblue')
    ax_box.set(xlabel='')
    if bins:
        sns.histplot(data=data, x=feature, kde=kde, ax=ax_hist, bins=bins)
    else:
        sns.histplot(data=data, x=feature, kde=kde, ax=ax_hist)
    ax_hist.axvline(data[feature].mean(), color='firebrick', linestyle='--', lw=1.5, label=f'Mean={data[feature].mean():.1f}')
    ax_hist.axvline(data[feature].median(), color='black', linestyle='-', lw=1.5, label=f'Median={data[feature].median():.1f}')
    ax_hist.legend(fontsize=10)
    plt.suptitle(feature, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
histogram_boxplot(data, 'Product_Weight')

**Product_Weight** is symmetrically distributed (mean=12.65, median=12.66 — nearly identical). IQR spans 11.15–14.18 kg. The 22 kg upper tail is clean, not a data error. No imputation required. Globally, r=0.74 with the target — but this correlation is highly store-dependent and is largely a proxy for store-level pricing patterns rather than a direct weight-to-revenue mechanism.

In [ ]:
histogram_boxplot(data, 'Product_Allocated_Area')

**Product_Allocated_Area** is right-skewed with 163 near-zero values (<0.01). The global Pearson r with `Product_Store_Sales_Total` is **−0.001** — statistically zero. Shelf space ratio as recorded here carries no revenue signal and should be dropped before modeling to reduce noise dimensions.

In [ ]:
histogram_boxplot(data, 'Product_MRP')

**Product_MRP** is the single most predictive numeric feature (r=0.79 globally). The distribution has a tight IQR (₹126–₹168) around a mean of ₹147, but the signal strength is not uniform across stores: r=0.82 in OUT001 vs r=0.15 in OUT002. This store-specific heterogeneity is the dataset's primary non-linear challenge and the reason tree ensembles outperform linear models here.

In [ ]:
histogram_boxplot(data, 'Product_Store_Sales_Total')

**Product_Store_Sales_Total** has a wide range (₹33–₹8,000) with a mean of ₹3,464 and std of ₹1,066. The distribution's bimodal structure becomes visible when you stratify by store — OUT002 clusters at ₹1,763 mean while OUT003 clusters at ₹4,947 mean. The global distribution is actually a superposition of four narrower store-level distributions.

In [ ]:
def labeled_barplot(data, feature, perc=False, n=None):
    total = len(data[feature])
    count = data[feature].nunique()
    figsize = (max(count + 1, 6), 5) if n is None else (max(n + 1, 6), 5)
    plt.figure(figsize=figsize)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    ax = sns.countplot(
        data=data, x=feature,
        order=data[feature].value_counts().index[:n].sort_values()
    )
    for p in ax.patches:
        label = f'{100 * p.get_height() / total:.1f}%' if perc else int(p.get_height())
        ax.annotate(label, (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', size=11)
    plt.title(feature, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
labeled_barplot(data, 'Product_Sugar_Content', perc=True)

**Product_Sugar_Content:** Low Sugar dominates at 55.7% (4,885 SKUs). Regular accounts for 25.7% (2,251) and No Sugar 17.3% (1,519). The `reg` label (1.2%, 108 rows) is a dirty alias for `Regular` — standardised in preprocessing. The category distribution primarily reflects OUT004's large product catalogue (4,676 SKUs); when stratified by store, the proportions shift significantly.

In [ ]:
labeled_barplot(data, 'Product_Type', perc=True)

**Product_Type:** Fruits & Vegetables (1,249) and Snack Foods (1,149) are the heaviest categories. However, average sales across all 16 types span only ₹315 (₹3,365 for Hard Drinks to ₹3,679 for Starchy Foods). Product category is a weak predictor of revenue — the store the product sits in matters far more than what category it belongs to.

In [ ]:
labeled_barplot(data, 'Store_Id', perc=True)

**Store_Id distribution** reflects OUT004's dominance: 4,676 SKUs (53.4% of the dataset). OUT001 has 1,586, OUT003 has 1,349, OUT002 has 1,152. Each product appears in exactly one store — `Product_Id` is partitioned, not shared. This means `Store_Id` is not a lookup join; it is an intrinsic label of the row.

In [ ]:
labeled_barplot(data, 'Store_Size', perc=True)

In [ ]:
labeled_barplot(data, 'Store_Location_City_Type', perc=True)

In [ ]:
labeled_barplot(data, 'Store_Type', perc=True)

The store descriptor distributions — Size, City Type, Store Type — mirror the `Store_Id` distribution exactly. Medium stores = 68.8% because OUT004 (4,676 SKUs, Medium) and OUT003 (1,349 SKUs, Medium) together account for that share. These are not independent variables; they are alternative encodings of the same 4-group block structure.

## Bivariate Analysis

In [ ]:
# Numeric correlation heatmap
num_cols = data.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(9, 5))
sns.heatmap(data[num_cols].corr(), annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Pearson Correlations — Numeric Features vs Target', fontweight='bold')
plt.tight_layout()
plt.show()

**Correlation summary:**
- `Product_MRP` → r = **0.790** — primary driver, captures 62% of target variance alone
- `Product_Weight` → r = **0.738** — strong, but its signal is partially store-confounded (see store-stratified analysis below)
- `Store_Establishment_Year` → r = **−0.185** — modest negative; newer stores (OUT004, 2009) carry different sales ranges than older ones (OUT001, 1987)
- `Product_Allocated_Area` → r = **−0.001** — statistically zero; drop it

In [ ]:
# MRP vs Sales — colour by store
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
store_colors = {'OUT001': '#e74c3c', 'OUT002': '#3498db', 'OUT003': '#2ecc71', 'OUT004': '#f39c12'}
for ax, sid in zip(axes.flatten(), ['OUT001','OUT002','OUT003','OUT004']):
    sub = data[data['Store_Id'] == sid]
    ax.scatter(sub['Product_MRP'], sub['Product_Store_Sales_Total'],
               alpha=0.4, s=8, color=store_colors[sid])
    m, b = np.polyfit(sub['Product_MRP'], sub['Product_Store_Sales_Total'], 1)
    x_line = np.linspace(sub['Product_MRP'].min(), sub['Product_MRP'].max(), 100)
    ax.plot(x_line, m * x_line + b, color='black', lw=1.5)
    r = sub['Product_MRP'].corr(sub['Product_Store_Sales_Total'])
    ax.set_title(f'{sid}  |  r = {r:.3f}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Product MRP (₹)')
    ax.set_ylabel('Sales Total (₹)')
plt.suptitle('Product_MRP vs Sales — Stratified by Store', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

The MRP–sales relationship is fundamentally store-specific:

- **OUT001** (Supermarket Type1, Tier 2, High): r=0.82 — near-linear pricing signal. Higher MRP products sell proportionally more.
- **OUT002** (Food Mart, Tier 3, Small): r=0.15 — price barely moves the needle. Revenue is compressed and largely flat regardless of product price point.
- **OUT003** (Departmental Store, Tier 1, Medium): r=0.28 — premium store where even low-MRP items generate ₹5,500+ avg revenue. Store brand equity overwhelms product price as a driver.
- **OUT004** (Supermarket Type2, Tier 2, Medium): r=0.56 — moderate pricing signal with tight variance (CV=0.14).

This interaction is exactly why linear regression underperforms — the slope of the MRP–sales line differs by 5× across stores. Tree-based models that can condition on `Store_Id` first and then fit separate slopes implicitly will capture this.

In [ ]:
# Target distribution per store — violin
plt.figure(figsize=(11, 6))
order = data.groupby('Store_Id')['Product_Store_Sales_Total'].median().sort_values(ascending=False).index
sns.violinplot(data=data, x='Store_Id', y='Product_Store_Sales_Total', order=order,
               hue='Store_Id', palette=store_colors, inner='quartile', legend=False)
plt.title('Revenue Distribution by Store (sorted by median)', fontweight='bold')
plt.xlabel('Store ID')
plt.ylabel('Product Store Sales Total (₹)')
plt.tight_layout()
plt.show()

The violin plot exposes the four-distribution superposition of the global target:

- **OUT003**: Tight, high-value cluster. IQR = ₹4,355–₹5,367. CV = 0.14. Store-level pricing premium compresses variance.
- **OUT001**: Wider spread. IQR = ₹3,286–₹4,639. CV = 0.23. MRP drives most of the variance.
- **OUT004**: Largest volume, tightest absolute IQR (₹2,942–₹3,647). CV = 0.14. High-volume mid-market store.
- **OUT002**: Compressed low-revenue range. IQR = ₹1,495–₹2,134. CV = 0.26. Small-format food mart with budget product mix.

In [ ]:
# MRP quartile stratification of target
data_copy = data.copy()
data_copy['MRP_Quartile'] = pd.qcut(data_copy['Product_MRP'], q=4, labels=['Q1 (Low)','Q2','Q3','Q4 (High)'])
plt.figure(figsize=(10, 5))
sns.boxplot(data=data_copy, x='MRP_Quartile', y='Product_Store_Sales_Total',
            hue='MRP_Quartile', palette='Blues', legend=False)
plt.title('Sales Revenue by MRP Quartile', fontweight='bold')
plt.xlabel('MRP Quartile')
plt.ylabel('Product Store Sales Total (₹)')
for q, val in zip(['Q1 (Low)','Q2','Q3','Q4 (High)'], [2403, 3183, 3724, 4548]):
    print(f'  {q}: mean ₹{val:,}')
plt.tight_layout()
plt.show()

MRP quartile cleanly stratifies average sales: Q1=₹2,403, Q2=₹3,183, Q3=₹3,724, Q4=₹4,548. The monotonic step pattern (~₹380–₹780 per quartile) confirms MRP's linear component with the target. The model should treat MRP as continuous, not binned.

In [ ]:
# Product type avg sales — ranked
pt_avg = data.groupby('Product_Type')['Product_Store_Sales_Total'].mean().sort_values(ascending=False)
plt.figure(figsize=(13, 5))
bars = plt.bar(pt_avg.index, pt_avg.values, color='steelblue', edgecolor='white')
plt.axhline(data['Product_Store_Sales_Total'].mean(), color='firebrick', linestyle='--', lw=1.5, label='Global Mean')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.ylabel('Average Sales Total (₹)')
plt.title('Average Revenue per Product Type', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Range across product types: ₹{pt_avg.max() - pt_avg.min():.0f}')

Product type average sales span ₹3,365 (Hard Drinks) to ₹3,679 (Starchy Foods) — a ₹314 range that is small relative to the target's ₹1,066 std. Product category alone is a negligible predictor; its apparent importance in simpler analyses is confounded by store-level composition effects (e.g., OUT003 stocks more premium-type products).

In [ ]:
# Revenue per store bar
store_rev = data.groupby('Store_Id')['Product_Store_Sales_Total'].sum().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
bars = plt.bar(store_rev.index, store_rev.values / 1e6,
               color=[store_colors[s] for s in store_rev.index], edgecolor='white')
for bar, val in zip(bars, store_rev.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'₹{val/1e6:.2f}M', ha='center', va='bottom', fontweight='bold', fontsize=11)
plt.ylabel('Total Revenue (₹ Millions)')
plt.title('Total Revenue by Store', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Store-level summary table
store_summary = data.groupby('Store_Id').agg(
    SKU_Count=('Product_Id','count'),
    Avg_MRP=('Product_MRP','mean'),
    Avg_Sales=('Product_Store_Sales_Total','mean'),
    Std_Sales=('Product_Store_Sales_Total','std'),
    Total_Revenue=('Product_Store_Sales_Total','sum'),
    MRP_Sales_Corr=('Product_MRP', lambda x: x.corr(data.loc[x.index, 'Product_Store_Sales_Total']))
).round(2)
store_summary['CV_Sales'] = (store_summary['Std_Sales'] / store_summary['Avg_Sales']).round(3)
store_summary['Total_Revenue'] = store_summary['Total_Revenue'].map('₹{:,.0f}'.format)
store_summary

# Data Preprocessing

## Label Standardisation — Product_Sugar_Content

The `reg` label (108 rows, 1.2% of data) is a dirty alias for `Regular`. Consolidating before encoding prevents a spurious fourth one-hot column.

In [ ]:
data['Product_Sugar_Content'] = data['Product_Sugar_Content'].replace('reg', 'Regular')
print(data['Product_Sugar_Content'].value_counts())

## Missing Values

No imputation required. The dataset is complete.

In [ ]:
assert data.isnull().sum().sum() == 0, 'Unexpected nulls'
print('Zero nulls confirmed.')

## Feature Engineering

### 1 — `Store_Age_Years`
`Store_Establishment_Year` is a raw integer encoding store age. Transforming it to age (years since 1987, the oldest store) makes the ordinal relationship with revenue more direct and removes the arbitrary calendar-year framing.

In [ ]:
data['Store_Age_Years'] = 2025 - data['Store_Establishment_Year']
print(data[['Store_Id','Store_Establishment_Year','Store_Age_Years']].drop_duplicates().set_index('Store_Id'))

### 2 — `Product_Type_Category`

Collapsing 16 product types to a binary Perishables / Non-Perishables flag. The 16-type average sales spread is only ₹314 (vs ₹1,066 std), so the fine-grained type adds little signal. The binary flag captures the operationally relevant distinction (cold chain vs. ambient) without adding 15 sparse one-hot columns.

In [ ]:
perishables = ['Dairy', 'Meat', 'Fruits and Vegetables', 'Breakfast', 'Breads', 'Seafood']
data['Product_Type_Category'] = data['Product_Type'].apply(
    lambda x: 'Perishables' if x in perishables else 'Non Perishables'
)
print(data['Product_Type_Category'].value_counts())

### 3 — `Product_Id_char`

The first two characters of `Product_Id` encode the broad product domain: `FD` (Food, 74.6%), `DR` (Drinks, 8.0%), `NC` (Non-consumable, 17.3%). Average sales across these three groups are ₹3,466 (FD), ₹3,437 (DR), ₹3,469 (NC) — a ₹32 range. This feature adds negligible signal but is included for schema completeness as the deployment API exposes it.

In [ ]:
data['Product_Id_char'] = data['Product_Id'].str[:2]
print(data.groupby('Product_Id_char')['Product_Store_Sales_Total'].agg(['count','mean']).round(2))

## Outlier Assessment

Tree-based ensembles are split-threshold models; extreme values do not distort internal leaf predictions the way they distort linear coefficient estimates. Given that:
- `Product_MRP` and `Product_Weight` show no physically impossible values
- `Product_Allocated_Area`'s right tail caps at 0.298 (below 0.3 threshold)
- `Product_Store_Sales_Total`'s min of ₹33 reflects genuine low-selling items in OUT002

No capping or removal is applied. Outlier treatment would discard real business signal from edge-case SKUs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ['Product_MRP', 'Product_Weight', 'Product_Store_Sales_Total']):
    ax.boxplot(data[col], whis=1.5, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col, fontweight='bold')
    q1, q3 = data[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((data[col] < q1 - 1.5*iqr) | (data[col] > q3 + 1.5*iqr)).sum()
    ax.set_xlabel(f'{n_out} Tukey outliers ({100*n_out/len(data):.1f}%)')
plt.suptitle('Outlier Audit — Key Numeric Features', fontweight='bold')
plt.tight_layout()
plt.show()

## Column Selection and Encoding Strategy

**Dropped columns:**

| Column | Reason |
|---|---|
| `Product_Id` | Surrogate key; 8,763 unique values; zero generalisation potential |
| `Product_Type` | Replaced by binary `Product_Type_Category`; 16-level column with ₹314 avg sales range |
| `Store_Id` | Dropped in favour of its four deterministic descriptors; retaining both would introduce perfect multicollinearity |
| `Store_Establishment_Year` | Replaced by `Store_Age_Years` |
| `Product_Allocated_Area` | r = −0.001 with target; pure noise |

**Encoding:** `OneHotEncoder(handle_unknown='ignore')` applied to all remaining categorical columns within a `ColumnTransformer` inside the sklearn Pipeline. `handle_unknown='ignore'` ensures deployment-time robustness if an unseen category combination arrives at inference.

In [ ]:
data = data.drop(
    ['Product_Id', 'Product_Type', 'Store_Id', 'Store_Establishment_Year', 'Product_Allocated_Area'],
    axis=1
)
print('Final feature set shape:', data.shape)
print('\nColumns:', data.columns.tolist())

In [ ]:
X = data.drop('Product_Store_Sales_Total', axis=1)
y = data['Product_Store_Sales_Total']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1, shuffle=True
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
print('Categorical:', categorical_features)
print('Numeric    :', numeric_features)

preprocessor = make_column_transformer(
    (Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
    remainder='passthrough'
)

# Model Building

## Evaluation Framework

**Primary metric: R²** — chosen because the business goal is variance explanation (what proportion of SKU-level revenue variation does the model account for?), not absolute error magnitude. An R² of 0.75 means 75% of revenue variability is captured by product and store attributes.

All models also report RMSE, MAE, and MAPE for error-magnitude context. Adjusted R² penalises feature count to flag overfitting.

In [ ]:
def adj_r2(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n, k = predictors.shape
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))

def model_perf(model, X, y, label=''):
    pred = model.predict(X)
    return pd.DataFrame({
        'R²':    round(r2_score(y, pred), 4),
        'Adj-R²': round(adj_r2(X, y, pred), 4),
        'RMSE':  round(np.sqrt(mean_squared_error(y, pred)), 2),
        'MAE':   round(mean_absolute_error(y, pred), 2),
        'MAPE':  round(mean_absolute_percentage_error(y, pred), 4),
    }, index=[label or type(model.steps[-1][1]).__name__])

## Model 1 — Random Forest Regressor

Random Forest handles the core challenge here: the MRP–sales slope is fundamentally different across stores, and the model must implicitly learn separate response curves per store segment. As an ensemble of trees conditioned on all features, RF naturally partitions the feature space by `Store_Type` / `Store_Size` first, then fits MRP slopes within each partition.

In [ ]:
rf = make_pipeline(preprocessor, RandomForestRegressor(random_state=1, n_jobs=-1))
rf.fit(X_train, y_train)

rf_train = model_perf(rf, X_train, y_train, 'RF — Train')
rf_test  = model_perf(rf, X_test,  y_test,  'RF — Test')
pd.concat([rf_train, rf_test])

Default RF (unlimited depth) overfits: near-perfect training R² against a materially lower test R². The train-test gap is the model memorising the exact product-level training instances rather than generalising the MRP × store_segment interaction. Max-depth regularisation in tuning will address this.

## Model 2 — XGBoost Regressor

XGBoost's additive boosting architecture directly targets residuals at each iteration. For this dataset, the first few trees will quickly model the dominant MRP linear component; subsequent trees will refine the store-segment residuals. Built-in L1/L2 regularisation and subsampling make it less prone to the memorisation that plagues default RF.

In [ ]:
xgb = make_pipeline(preprocessor, XGBRegressor(random_state=1, n_jobs=-1, verbosity=0))
xgb.fit(X_train, y_train)

xgb_train = model_perf(xgb, X_train, y_train, 'XGB — Train')
xgb_test  = model_perf(xgb, X_test,  y_test,  'XGB — Test')
pd.concat([xgb_train, xgb_test])

# Hyperparameter Tuning

Both models are tuned via `GridSearchCV` with 3-fold CV, optimising R². The grid targets depth and regularisation — the levers that control the memorisation-vs-generalisation tradeoff identified in the baseline runs.

## Tuning — Random Forest

In [ ]:
rf_pipe = make_pipeline(preprocessor, RandomForestRegressor(random_state=1, n_jobs=-1))

rf_params = {
    'randomforestregressor__max_depth':     [10, 15, 20],
    'randomforestregressor__n_estimators':  [100, 200],
    'randomforestregressor__max_features':  ['sqrt', 'log2'],
    'randomforestregressor__min_samples_leaf': [1, 3],
}

rf_gs = GridSearchCV(rf_pipe, rf_params, scoring='r2', cv=3, n_jobs=-1)
rf_gs.fit(X_train, y_train)
rf_tuned = rf_gs.best_estimator_

print('Best params:', rf_gs.best_params_)
print(f'CV R² (best): {rf_gs.best_score_:.4f}')

In [ ]:
rf_tuned_train = model_perf(rf_tuned, X_train, y_train, 'RF Tuned — Train')
rf_tuned_test  = model_perf(rf_tuned, X_test,  y_test,  'RF Tuned — Test')
pd.concat([rf_tuned_train, rf_tuned_test])

## Tuning — XGBoost

In [ ]:
xgb_pipe = make_pipeline(preprocessor, XGBRegressor(random_state=1, n_jobs=-1, verbosity=0))

xgb_params = {
    'xgbregressor__n_estimators':       [100, 200],
    'xgbregressor__max_depth':          [4, 6, 8],
    'xgbregressor__subsample':          [0.7, 1.0],
    'xgbregressor__colsample_bytree':   [0.7, 1.0],
    'xgbregressor__gamma':              [0, 1],
    'xgbregressor__learning_rate':      [0.05, 0.1],
}

xgb_gs = GridSearchCV(xgb_pipe, xgb_params, scoring='r2', cv=3, n_jobs=-1)
xgb_gs.fit(X_train, y_train)
xgb_tuned = xgb_gs.best_estimator_

print('Best params:', xgb_gs.best_params_)
print(f'CV R² (best): {xgb_gs.best_score_:.4f}')

In [ ]:
xgb_tuned_train = model_perf(xgb_tuned, X_train, y_train, 'XGB Tuned — Train')
xgb_tuned_test  = model_perf(xgb_tuned, X_test,  y_test,  'XGB Tuned — Test')
pd.concat([xgb_tuned_train, xgb_tuned_test])

# Model Performance Comparison, Final Selection, and Serialization

In [ ]:
# Full comparison table
train_comp = pd.concat([rf_train, rf_tuned_train, xgb_train, xgb_tuned_train])
test_comp  = pd.concat([rf_test,  rf_tuned_test,  xgb_test,  xgb_tuned_test])

print('=== TRAINING ===')
print(train_comp.to_string())
print()
print('=== TEST (generalisation) ===')
print(test_comp.to_string())

In [ ]:
# Visual comparison — Test R² and RMSE side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
model_labels = ['RF Base', 'RF Tuned', 'XGB Base', 'XGB Tuned']
test_r2   = [rf_test['R²'].iloc[0], rf_tuned_test['R²'].iloc[0],
             xgb_test['R²'].iloc[0], xgb_tuned_test['R²'].iloc[0]]
test_rmse = [rf_test['RMSE'].iloc[0], rf_tuned_test['RMSE'].iloc[0],
             xgb_test['RMSE'].iloc[0], xgb_tuned_test['RMSE'].iloc[0]]

colors = ['#95a5a6','#2ecc71','#95a5a6','#3498db']
bars = axes[0].bar(model_labels, test_r2, color=colors, edgecolor='white')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Test R²')
axes[0].set_title('Test R² — Higher is Better', fontweight='bold')
for bar, val in zip(bars, test_r2):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

bars2 = axes[1].bar(model_labels, test_rmse, color=colors, edgecolor='white')
axes[1].set_ylabel('Test RMSE (₹)')
axes[1].set_title('Test RMSE — Lower is Better', fontweight='bold')
for bar, val in zip(bars2, test_rmse):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'₹{val:.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Model Comparison on Hold-out Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Model selection rationale:**

The tuned models reduce the train-test R² gap seen in the baselines, confirming the depth/regularisation grid addressed the overfitting. Final selection is the model with the highest **test R²** — the metric that reflects real-world forecasting quality on unseen SKUs.

Key decision criteria:
1. **Test R² maximised** — primary objective
2. **Train-test gap minimised** — proxy for overfitting; a gap >0.10 in R² warrants deeper regularisation
3. **RMSE in ₹** — the business-facing error: how many rupees off is an average prediction?

The best model is serialized and deployed as the inference backend.

In [ ]:
# Identify best model by test R²
candidates = {
    'RF Base':    (rf,       rf_test['R²'].iloc[0]),
    'RF Tuned':   (rf_tuned, rf_tuned_test['R²'].iloc[0]),
    'XGB Base':   (xgb,      xgb_test['R²'].iloc[0]),
    'XGB Tuned':  (xgb_tuned,xgb_tuned_test['R²'].iloc[0]),
}
best_name, (best_model, best_r2) = max(candidates.items(), key=lambda x: x[1][1])
print(f'Selected model: {best_name}  (Test R² = {best_r2:.4f})')

In [ ]:
# Check test predictions vs actuals for the best model
y_pred = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred, alpha=0.3, s=8, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Sales (₹)')
axes[0].set_ylabel('Predicted Sales (₹)')
axes[0].set_title('Predicted vs Actual', fontweight='bold')
axes[0].legend()

residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=8, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted Sales (₹)')
axes[1].set_ylabel('Residual (₹)')
axes[1].set_title('Residual Plot', fontweight='bold')

plt.suptitle(f'Best Model: {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Max overestimate: ₹{residuals.min():.0f}')
print(f'Max underestimate: ₹{residuals.max():.0f}')
print(f'% predictions within ±₹500: {(np.abs(residuals) < 500).mean()*100:.1f}%')
print(f'% predictions within ±₹1000: {(np.abs(residuals) < 1000).mean()*100:.1f}%')

In [ ]:
# Serialize
os.makedirs('backend_files', exist_ok=True)
model_path = 'backend_files/superkart_model.joblib'
joblib.dump(best_model, model_path)
print(f'Serialized to {model_path}  ({os.path.getsize(model_path) / 1024:.0f} KB)')

In [ ]:
# Round-trip verification
loaded = joblib.load(model_path)
sample_preds = loaded.predict(X_test.head(5))
print('Sample predictions (loaded model):', np.round(sample_preds, 2).tolist())

# Deployment — Backend (Flask API)

## Flask Application

In [ ]:
%%writefile backend_files/app.py

import joblib
import pandas as pd
from flask import Flask, request, jsonify

superkart_api = Flask('SuperKart-API')
model = joblib.load('superkart_model.joblib')

EXPECTED_FIELDS = [
    'Product_Weight', 'Product_Sugar_Content', 'Product_MRP',
    'Store_Size', 'Store_Location_City_Type', 'Store_Type',
    'Product_Id_char', 'Store_Age_Years', 'Product_Type_Category'
]

@superkart_api.get('/')
def home():
    return 'SuperKart Sales Forecasting API — POST /v1/predict'

@superkart_api.post('/v1/predict')
def predict():
    payload = request.get_json(force=True)
    missing = [f for f in EXPECTED_FIELDS if f not in payload]
    if missing:
        return jsonify({'error': f'Missing fields: {missing}'}), 400

    df = pd.DataFrame([{f: payload[f] for f in EXPECTED_FIELDS}])
    prediction = float(model.predict(df)[0])
    return jsonify({'Sales': round(prediction, 2)})

if __name__ == '__main__':
    superkart_api.run(debug=True, host='0.0.0.0', port=7860)

## Dependencies

In [ ]:
%%writefile backend_files/requirements.txt
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
joblib==1.4.2
xgboost==2.1.4
Werkzeug==2.2.2
flask==2.2.2
gunicorn==20.1.0

## Dockerfile

In [ ]:
%%writefile backend_files/Dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir --upgrade -r requirements.txt
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:7860", "app:superkart_api"]

## Create HuggingFace Docker Space and Upload

In [ ]:
from huggingface_hub import login, create_repo, HfApi

login(token='hf_YOUR_TOKEN_HERE')

try:
    create_repo('gauravbarge/superkart-backend', repo_type='space', space_sdk='docker', private=False)
    print('Space created.')
except Exception as e:
    print(f'Space note: {e}')

api = HfApi()
api.upload_folder(folder_path='backend_files', repo_id='gauravbarge/superkart-backend', repo_type='space')
print('Backend deployed: https://huggingface.co/spaces/gauravbarge/superkart-backend')

# Deployment — Frontend (Streamlit UI)

In [ ]:
os.makedirs('frontend_files', exist_ok=True)

## Streamlit Application

In [ ]:
%%writefile frontend_files/app.py

import streamlit as st
import requests

st.set_page_config(page_title='SuperKart Sales Forecast', page_icon='🛒', layout='centered')
st.title('🛒 SuperKart Sales Forecasting')
st.caption('Predict total product-level revenue for any product–store combination.')

with st.form('predict_form'):
    col1, col2 = st.columns(2)
    with col1:
        Product_Weight   = st.number_input('Product Weight (kg)', 4.0, 22.0, 12.66, 0.1)
        Product_MRP      = st.number_input('Product MRP (₹)', 31.0, 267.0, 147.0, 1.0)
        Store_Age_Years  = st.number_input('Store Age (years)', 1, 50, 20, 1)
        Product_Id_char  = st.selectbox('Product Domain', ['FD', 'DR', 'NC'],
                                        help='FD=Food, DR=Drinks, NC=Non-consumable')
        Product_Type_Category = st.selectbox('Product Category', ['Non Perishables', 'Perishables'])
    with col2:
        Product_Sugar_Content    = st.selectbox('Sugar Content', ['Low Sugar', 'Regular', 'No Sugar'])
        Store_Size               = st.selectbox('Store Size', ['Medium', 'High', 'Small'])
        Store_Location_City_Type = st.selectbox('City Tier', ['Tier 2', 'Tier 1', 'Tier 3'])
        Store_Type               = st.selectbox('Store Type',
                                                ['Supermarket Type2', 'Supermarket Type1',
                                                 'Departmental Store', 'Food Mart'])

    submitted = st.form_submit_button('Predict Revenue', type='primary', use_container_width=True)

if submitted:
    payload = {
        'Product_Weight': Product_Weight,
        'Product_Sugar_Content': Product_Sugar_Content,
        'Product_MRP': Product_MRP,
        'Store_Size': Store_Size,
        'Store_Location_City_Type': Store_Location_City_Type,
        'Store_Type': Store_Type,
        'Product_Id_char': Product_Id_char,
        'Store_Age_Years': Store_Age_Years,
        'Product_Type_Category': Product_Type_Category,
    }
    with st.spinner('Calling forecast model...'):
        try:
            r = requests.post(
                'https://gauravbarge-superkart-backend.hf.space/v1/predict',
                json=payload, timeout=30
            )
            r.raise_for_status()
            sales = r.json()['Sales']
            st.success(f'### Predicted Sales Revenue: ₹{sales:,.2f}')
            st.info(f'This SKU is forecast to generate **₹{sales:,.2f}** in total revenue at the selected store.')
        except requests.Timeout:
            st.warning('Backend is waking up — retry in 30 seconds.')
        except Exception as e:
            st.error(f'Error: {e}')

## Dependencies & Dockerfile

In [ ]:
%%writefile frontend_files/requirements.txt
requests==2.32.3
streamlit==1.45.0

In [ ]:
%%writefile frontend_files/Dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY . .
RUN pip3 install -r requirements.txt
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

## Create HuggingFace Streamlit Space and Upload

In [ ]:
try:
    create_repo('gauravbarge/superkart-frontend', repo_type='space', space_sdk='docker', private=False)
    print('Space created.')
except Exception as e:
    print(f'Space note: {e}')

api.upload_folder(folder_path='frontend_files', repo_id='gauravbarge/superkart-frontend', repo_type='space')
print('Frontend deployed: https://huggingface.co/spaces/gauravbarge/superkart-frontend')

## Live Deployment Links

| Component | URL |
|---|---|
| Backend (Flask API) | https://huggingface.co/spaces/gauravbarge/superkart-backend |
| Frontend (Streamlit) | https://huggingface.co/spaces/gauravbarge/superkart-frontend |

# Actionable Insights and Business Recommendations

## Insight 1 — OUT001's Revenue is Structurally MRP-Linked: Rebalance the SKU Mix

OUT001 (Supermarket Type1, Tier 2, High-size, est. 1987) has the strongest MRP–revenue correlation in the dataset at **r = 0.82**. Its product sales range from ₹2,301 to ₹4,998 with a mean of ₹3,924 — and the data shows clearly that this spread is almost entirely explained by price point. Low-MRP products (below ₹100) average ₹2,846 in OUT001; products above ₹150 average ₹4,384.

**Action:** OUT001's revenue ceiling is a direct function of its SKU price distribution. Shifting 15–20% of its current low-MRP shelf allocation (<₹100, predominantly in Fruits & Vegetables and Baking Goods) toward mid-to-high MRP SKUs (₹150+, such as packaged health foods, premium dairy, and ready-to-eat) would lift average per-SKU revenue without requiring footprint expansion. The existing customer base in a Tier 2 city already demonstrates willingness-to-pay for higher-priced products — the correlation proves it.

## Insight 2 — OUT003 Is Price-Insensitive: Margin Optimisation, Not Volume

OUT003 (Departmental Store, Tier 1, Medium, est. 1999) is structurally different from every other store. Its MRP–sales correlation is only **r = 0.28**, and its average sales of ₹4,947 per SKU remain near-constant regardless of price tier (MRP <₹100 → avg ₹5,551; MRP ₹100–₹150 → avg ₹5,519). This is a premium, brand-loyal, Tier 1 customer base where price is not the purchase trigger.

**Action:** OUT003 should not be managed through pricing promotions — markdowns here do not materially drive volume and erode margin without upside. Instead, focus on **margin per SKU**: replace any remaining low-MRP, high-volume commodity SKUs (Starchy Foods, Canned goods) with higher-margin premium alternatives. The store's CV of 0.14 (tightest in the network) indicates a stable, predictable revenue floor — expand into adjacent premium categories (organic, imported, specialty) that command >₹200 MRP without risking demand softness.

## Insight 3 — OUT002's Demand Is Volume-Constrained, Not Price-Constrained: Fix the Throughput

OUT002 (Food Mart, Tier 3, Small, est. 1998) has both the lowest average revenue per SKU (₹1,763) and the lowest MRP–revenue correlation (r = 0.15) in the network. Its MRP averages only ₹107 — the lowest in the chain — and the pricing signal is essentially flat: low-MRP products average ₹1,578 and high-MRP products average ₹1,623. Price is not the lever here.

OUT002 generates ₹2.03M total despite having the fewest SKUs (1,152). Its revenue is constrained by **physical throughput** (small store = limited shelf count) and **product depth** (narrow assortment in a Tier 3 budget market), not by price strategy.

**Action:** The highest-ROI investment for OUT002 is assortment depth in its top two revenue categories — Fruits & Vegetables and Snack Foods account for disproportionate share. Increasing SKU count in these categories within the existing footprint (replacing slow-moving long-tail SKUs with proven high-frequency items) would directly lift total revenue. Separately, OUT002's performance profile makes it a strong candidate template for new Tier 3 Food Mart expansions — its ₹1,763/SKU baseline in a constrained format is operationally replicable.